# DWARF-R1 C5 paired experiment — preflight only

**This notebook has no training or inference cell.** It reads Drive artifacts, audits the embedded C4 data, reports dependency/runtime/GPU status, and writes only a small preflight JSON to the ephemeral Colab `/content` directory. It never changes Drive data, model files, checkpoints, or Factory state.

## Intended comparison
- **A:** verified C4 adapter + original C4 training data/recipe.
- **B:** the same C4 adapter + the same core data and budget, with approved Factory Manager additions.
- Paired seeds requested: `182032`, `182033`, `182034`; historical exposure is recorded, not treated as an automatic failure.
- Metrics: accuracy, macro-F1, false acceptance, Work ID/Receipt validation, queue-decision accuracy, with per-seed and aggregate mean/variance. Accuracy ≥85% is necessary but insufficient; any unauthorized false acceptance or failed deterministic validation blocks acceptance.

## Source findings already identified
- Drive search found the `DWARF_R1_Factory_Manager.ipynb`, but no separate Factory Manager additions dataset or approved manifest by those names. The notebook contains historical synthetic data recipes; those are not presumed approved C5 additions.
- Two `boundary_dataset_receipt.json` files were found. Their manifests identify HERMES Academy scope-gate fixtures with in-memory/no-live-integration scope; they are not DWARF Factory Manager data and will not be used.
- C4’s recorded split counts are 512/128/128 and its manifest says 64 synthetic policy families are shared across splits. Different IDs/layouts do not demonstrate family-level isolation.
- Historical training evidence: C4 used seed `182032`; a prior single-run C5 candidate used seed `182033`. Seed `182034` remains unknown from the evidence checked so far. These facts are reported without changing the requested seed list.
- Light Bookmarks can read/list the C4 archive metadata; it does not expose Drive ACL details. A live hash below verifies the bytes only, not the owner/sharing ACL.

**Do not train from this notebook.** If no approved Factory Manager dataset/manifest is found, the C5 data gate remains BLOCKED. Any future holdout must be frozen with auditable provenance and grouped away from training. Synthetic evaluation is not Android or Production acceptance.


In [ ]:
# Protocol constants only. No training is implemented in this notebook.
from pathlib import Path
import hashlib, json, zipfile, subprocess, sys

C4_ARCHIVE_NAME = "DWARF_R1_candidate4_model_and_evidence.zip"
C4_ARCHIVE_SIZE = 71_172_139
C4_ARCHIVE_SHA256 = "a23d3ae12e235f4cfe559848525179745bd040a2249661dd2290b3fe0cfdb4ff"
C4_ADAPTER_SHA256 = "f36716a0e1d9154abfa1aee5f61b418394745fccc8bb8e3c561d2941fe43e3aa"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
REQUESTED_SEEDS = (182032, 182033, 182034)
HISTORICALLY_EXPOSED_SEEDS = {182032, 182033}
EXPECTED_C4_COUNTS = {"train": 512, "validation": 128, "test": 128}
EXPECTED_C4_DATA_SHA256 = {
    "train": "0c29ea219992e038060f7d6eea6c26dfc7dbbb44937b3aa122267ce2d5fd2dba",
    "validation": "3041aad4847f0acd8969bad5996774b1e57b1afbc9118a570ee63ee34a2fa055",
    "test": "0dd180696efa851f2939ad46fc28e7a292f980d89f6d02eca759fee16f6ac538",
}

# C4 recipe recorded in the Factory Manager notebook: weights-only continuation, fresh optimizer.
REFERENCE_RECIPE = {
    "learning_rate": 1e-5, "optimizer": "AdamW", "weight_decay": 0.01,
    "train_rows": 512, "optimizer_steps": 128, "gradient_accumulation": 4,
    "per_example_microbatch": 1, "max_grad_norm": 1.0, "precision": "fp16",
    "grad_scaler_init_scale": 256, "gradient_checkpointing": True,
    "base_model": BASE_MODEL, "base_revision": BASE_REVISION,
}
TRAINING_ENABLED = False
HUMAN_APPROVAL_RECORDED = False
FACTORY_ADDITIONS_PATH = None  # Set only to a separately reviewed, approved dataset file.
print("PREPARATION ONLY — training enabled:", TRAINING_ENABLED)
print("Requested seeds:", REQUESTED_SEEDS, "| historically exposed:", sorted(HISTORICALLY_EXPOSED_SEEDS))
print("Reference recipe:", json.dumps(REFERENCE_RECIPE, indent=2))


In [ ]:
# Read existing Drive artifacts. This cell does not write to Drive.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
C4_ARCHIVE = DRIVE_ROOT / C4_ARCHIVE_NAME
if not C4_ARCHIVE.is_file():
    candidates = list(DRIVE_ROOT.glob(C4_ARCHIVE_NAME))
    if len(candidates) == 1:
        C4_ARCHIVE = candidates[0]
    else:
        raise FileNotFoundError(f"Could not uniquely locate {C4_ARCHIVE_NAME} at MyDrive root; found {len(candidates)}")
print("C4 archive located:", C4_ARCHIVE)


In [ ]:
# Verify current archive bytes, ZIP CRC, internal file hashes, adapter, and dataset files.
def sha256_stream(stream):
    h = hashlib.sha256()
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        h.update(chunk)
    return h.hexdigest()

archive_sha = sha256_stream(C4_ARCHIVE.open("rb"))
archive_bytes = C4_ARCHIVE.stat().st_size
assert archive_bytes == C4_ARCHIVE_SIZE, (archive_bytes, C4_ARCHIVE_SIZE)
assert archive_sha == C4_ARCHIVE_SHA256, (archive_sha, C4_ARCHIVE_SHA256)
with zipfile.ZipFile(C4_ARCHIVE) as z:
    assert z.testzip() is None, "ZIP CRC failure"
    members = set(z.namelist())
    assert "SHA256SUMS.json" in members and "manifest.json" in members
    sums = json.loads(z.read("SHA256SUMS.json"))
    internal_bad = []
    for name, expected in sums.items():
        if name not in members:
            internal_bad.append((name, "missing")); continue
        actual = sha256_stream(z.open(name))
        if actual != expected:
            internal_bad.append((name, actual, expected))
    assert not internal_bad, internal_bad
    assert len(sums) == 39 and len(members) == 40, (len(sums), len(members))
    adapter_name = "adapter/adapter_model.safetensors"
    assert adapter_name in members
    adapter_sha = sha256_stream(z.open(adapter_name))
    assert adapter_sha == C4_ADAPTER_SHA256, (adapter_sha, C4_ADAPTER_SHA256)
    c4_manifest = json.loads(z.read("manifest.json"))
    split_bytes = {s: z.read(f"{s}.jsonl") for s in EXPECTED_C4_COUNTS}

print("ARCHIVE_SHA256_PASS", archive_sha, "bytes", archive_bytes)
print("ZIP_CRC_PASS; internal files:", len(sums), "/", len(members))
print("C4_ADAPTER_SHA256_PASS", adapter_sha)
print("C4 manifest counts:", c4_manifest.get("counts"))


In [ ]:
# Audit split bytes, row counts, exact ID reuse, and group overlap without printing examples.
import re
def parse_jsonl(blob):
    return [json.loads(line) for line in blob.decode("utf-8").splitlines() if line.strip()]

rows_by_split = {s: parse_jsonl(b) for s, b in split_bytes.items()}
for split, rows in rows_by_split.items():
    digest = hashlib.sha256(split_bytes[split]).hexdigest()
    print(split, "rows=", len(rows), "sha256=", digest, "expected=", EXPECTED_C4_DATA_SHA256[split])
    assert len(rows) == EXPECTED_C4_COUNTS[split]
    assert digest == EXPECTED_C4_DATA_SHA256[split]
assert c4_manifest.get("counts") == EXPECTED_C4_COUNTS

def index_splits(key_fn):
    index = {}
    for split, rows in rows_by_split.items():
        for row in rows:
            value = key_fn(row)
            if value not in (None, ""):
                index.setdefault(str(value), set()).add(split)
    return index

def overlap_report(index):
    pair_counts = {"train-validation": 0, "train-test": 0, "validation-test": 0}
    for splits in index.values():
        for pair in pair_counts:
            left, right = pair.split("-")
            if {left, right}.issubset(splits): pair_counts[pair] += 1
    triple = sum(splits == {"train", "validation", "test"} for splits in index.values())
    return {"unique_groups": len(index), "overlap_by_pair": pair_counts, "in_all_three": triple}

row_ids = index_splits(lambda r: r.get("id"))
work_ids = index_splits(lambda r: r.get("snapshot", {}).get("workId"))
evidence_refs = index_splits(lambda r: "|".join(sorted(r.get("snapshot", {}).get("evidenceRefs", []))))
prompts = index_splits(lambda r: hashlib.sha256(r.get("input", "").encode()).hexdigest())
exact_leakage = {
    "row_ids": overlap_report(row_ids), "work_ids": overlap_report(work_ids),
    "evidence_refs": overlap_report(evidence_refs), "identical_prompt_hashes": overlap_report(prompts),
}
print("EXACT_REUSE_AUDIT", json.dumps(exact_leakage, sort_keys=True))
exact_overlap_pass = all(not any(v for v in rep["overlap_by_pair"].values()) for rep in exact_leakage.values())

# `familyId` is included as reported, but may encode a split-specific layout rather than source family.
family_ids = index_splits(lambda r: r.get("familyId"))
category_groups = index_splits(lambda r: r.get("category"))
family_id_overlap = overlap_report(family_ids)
category_group_overlap = overlap_report(category_groups)
print("ROW_FAMILY_ID_OVERLAP", json.dumps(family_id_overlap, sort_keys=True))
print("CATEGORY_GROUP_OVERLAP_PROXY", json.dumps(category_group_overlap, sort_keys=True))
limitation = str(c4_manifest.get("limitation", ""))
reported = re.search(r"(\d+)\s+shared synthetic policy families", limitation, flags=re.I)
reported_shared_families = int(reported.group(1)) if reported else None
print("MANIFEST_FAMILY_LIMITATION", limitation)
print("MANIFEST_REPORTED_SHARED_POLICY_FAMILIES", reported_shared_families)
# Category/group-ID overlap is reported, not accepted as proof of group independence.
family_isolation_verified = False if reported_shared_families else False
print("FAMILY_LEVEL_SPLIT_ISOLATION: BLOCKED — source manifest reports shared synthetic policy families")


In [ ]:
# Record Python and installed package versions only; no installs and no model loading.
import importlib.metadata as metadata, platform
package_names = ["torch", "transformers", "peft", "accelerate", "safetensors", "google-api-python-client"]
package_versions = {}
for name in package_names:
    try: package_versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError: package_versions[name] = None
python_runtime = {"python": sys.version, "platform": platform.platform(), "packages": package_versions}
print("PYTHON_RUNTIME", json.dumps(python_runtime, sort_keys=True))


In [ ]:
# Fresh GPU probe. Historical notebook output is not reused as current evidence.
nvidia = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)
print("nvidia-smi return code:", nvidia.returncode)
print(nvidia.stdout.strip() or nvidia.stderr.strip())
try:
    import torch
    gpu_ready = bool(torch.cuda.is_available() and torch.cuda.device_count() > 0)
    print("torch:", torch.__version__, "cuda_available:", torch.cuda.is_available())
    if gpu_ready:
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except Exception as exc:
    gpu_ready = False
    print("torch/GPU probe failed:", repr(exc))
print("GPU_PREFLIGHT:", "PASS" if gpu_ready and nvidia.returncode == 0 else "BLOCKED — no live GPU evidence")


In [ ]:
# Gate summary. This notebook cannot start training, even if every probe passes.
seed_provenance = {
    182032: "USED — historical C4 training seed",
    182033: "USED — historical single-run C5 candidate seed",
    182034: "UNKNOWN — no prior use found in inspected records",
}
seed_audit_recorded = set(seed_provenance) == set(REQUESTED_SEEDS)
additions_present = bool(FACTORY_ADDITIONS_PATH and Path(FACTORY_ADDITIONS_PATH).is_file())
new_holdout_ready = False
checkpoint_ready = (archive_sha == C4_ARCHIVE_SHA256 and adapter_sha == C4_ADAPTER_SHA256)
gates = {
    "checkpoint_and_sha256": checkpoint_ready,
    "embedded_c4_split_integrity": True,
    "no_exact_cross_split_reuse": exact_overlap_pass,
    "approved_factory_manager_dataset_and_manifest": additions_present,
    "fresh_family_disjoint_holdout": new_holdout_ready,
    "seed_provenance_recorded": seed_audit_recorded,
    "live_colab_gpu_probe_completed": bool(nvidia.returncode == 0 or not gpu_ready),
    "human_training_approval_recorded": HUMAN_APPROVAL_RECORDED,
}
preflight_report = {
    "status": "PASS" if all(gates.values()) else "BLOCKED",
    "gates": gates,
    "archive": {"bytes": archive_bytes, "sha256": archive_sha, "zip_entries": len(members), "internal_sha256_files": len(sums), "adapter_sha256": adapter_sha},
    "dataset": {"counts": {s: len(rows) for s, rows in rows_by_split.items()}, "sha256": {s: hashlib.sha256(split_bytes[s]).hexdigest() for s in split_bytes}, "exact_reuse": exact_leakage, "familyId_overlap": family_id_overlap, "category_group_overlap_proxy": category_group_overlap, "manifest_reported_shared_families": reported_shared_families, "approved_additions_found": additions_present},
    "seeds": {str(k): v for k, v in seed_provenance.items()},
    "runtime": {"python": python_runtime, "nvidia_smi_returncode": nvidia.returncode, "nvidia_smi": nvidia.stdout.strip() or nvidia.stderr.strip(), "gpu_ready": bool(gpu_ready)},
    "training_started": False,
}
report_path = Path('/content/DWARF_R1_C5_preflight_report.json')
report_path.write_text(json.dumps(preflight_report, indent=2))
print(json.dumps(preflight_report, indent=2))
print("REPORT_SAVED_EPHEMERALLY", report_path)
print("TRAINING_ENABLED:", TRAINING_ENABLED, "— no training code exists in this notebook")
